## Visual Representation

<div align="center">
  <img src="../data/reliable_rag.svg" width="300" height="700" alt="Reliable RAG">
</div>


```bash
# Install required packages
!pip install langchain langchain-community python-dotenv
```


In [6]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")

GROQ_MODEL = os.getenv(
    "GROQ_MODEL",
    "llama-3.1-8b-instant"
)

## Create Vectorstore


In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = os.getenv("HUGGINGFACE_MODEL")

huggingface_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Docs to index
urls = [
    "https://www.deeplearning.ai/the-batch/how-agents-can-improve-llm-performance/?ref=dl-staging-website.ghost.io",
    "https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-2-reflection/?ref=dl-staging-website.ghost.io",
    "https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-3-tool-use/?ref=dl-staging-website.ghost.io",
    "https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-4-planning/?ref=dl-staging-website.ghost.io",
    "https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-5-multi-agent-collaboration/?ref=dl-staging-website.ghost.io"
]

docs = WebBaseLoader(urls).load()

# text splitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,
    chunk_overlap=0
)

doc_splits = text_splitter.split_documents(docs)

# Add to vectorStore
vectorStore = Chroma.from_documents(
    documents=doc_splits,
    collection_name="rag",
    embedding=huggingface_model,
)

retriever = vectorStore.as_retriever(
    search_type= "similarity",
    search_kwargs= {'k': 4} # number of documents to retrieve
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9965.25it/s]


In [24]:
question = "what are the differnt kind of agentic design patterns?"

In [25]:
print(f"Title: {docs[0].metadata['title']}\n\nSource: {docs[0].metadata['source']}\n\nContent: {docs[0].page_content}\n")

Title: Four AI Agent Strategies That Improve GPT-4 and GPT-3.5 Performance

Source: https://www.deeplearning.ai/the-batch/how-agents-can-improve-llm-performance/?ref=dl-staging-website.ghost.io

Content: Four AI Agent Strategies That Improve GPT-4 and GPT-3.5 Performance✨ New Course! Enroll in Building Adaptive AI AgentsCoursesNewsThe BatchAndrew's LetterData PointsML ResearchBlogCommunityForumEventsAmbassadorsAmbassador SpotlightResourcesMembershipFor BusinessOverviewPlans & PricingLearning TracksContact UsMembershipFor BusinessOverviewPlans & PricingLearning TracksContact UsCoursesNewsThe BatchAndrew's LetterData PointsML ResearchBlogCommunityForumEventsAmbassadorsAmbassador SpotlightResourcesMembershipFor BusinessOverviewPlans & PricingLearning TracksContact UsWeekly IssuesAndrew's LettersData PointsML ResearchBusinessScienceCultureHardwareAI CareersAboutSubscribeThe BatchLettersArticleAgentic Design Patterns Part 1 Four AI agent strategies that improve GPT-4 and GPT-3.5 performance

## Check document relevancy

In [26]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq

# Data model

class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved documents."""

    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )

# LLM with function call
llm = ChatGroq(model=GROQ_MODEL, temperature=0)
structure_llm_grader = llm.with_structured_output(GradeDocuments)

# Prompt
System = """
You are a grader assessing relevance of a retrieved document to a user question. \n 
    If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n
    It does not need to be a stringent test. The goal is to filter out erroneous retrievals. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question.
"""
grade_prompt = ChatPromptTemplate.from_messages([
    ("system", System),
    ("human", "Retrieved document: \n\n {document} \n\n User question: {question}"),
])

retrieval_grader = grade_prompt | structure_llm_grader

## Filter out the non-relevant docs

In [28]:
docs_to_use = []

for doc in docs:
    print(doc.page_content, '\n', '-' * 50)

    res = retrieval_grader.invoke({
        "question": question,
        "document": doc.page_content
    })

    print(res, '\n')

    if res.binary_score == "yes":
        docs_to_use.append(doc)

print("Relevant documents:", len(docs_to_use))

Four AI Agent Strategies That Improve GPT-4 and GPT-3.5 Performance✨ New Course! Enroll in Building Adaptive AI AgentsCoursesNewsThe BatchAndrew's LetterData PointsML ResearchBlogCommunityForumEventsAmbassadorsAmbassador SpotlightResourcesMembershipFor BusinessOverviewPlans & PricingLearning TracksContact UsMembershipFor BusinessOverviewPlans & PricingLearning TracksContact UsCoursesNewsThe BatchAndrew's LetterData PointsML ResearchBlogCommunityForumEventsAmbassadorsAmbassador SpotlightResourcesMembershipFor BusinessOverviewPlans & PricingLearning TracksContact UsWeekly IssuesAndrew's LettersData PointsML ResearchBusinessScienceCultureHardwareAI CareersAboutSubscribeThe BatchLettersArticleAgentic Design Patterns Part 1 Four AI agent strategies that improve GPT-4 and GPT-3.5 performanceLettersTechnical InsightsPublishedMar 20, 2024Reading time2 min readShareDear friends,I think AI agent workflows will drive massive AI progress this year — perhaps even more than the next generation of fo

## Generate Result

In [30]:
from langchain_core.output_parsers import StrOutputParser

# Prompt
system = """You are an assistant for question-answering tasks. Answer the question based upon your knowledge. 
Use three-to-five sentences maximum and keep the answer concise."""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved documents: \n\n <docs>{documents}</docs> \n\n User question: <question>{question}</question>"),
    ]
)

# LLM
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

# Post-processing
def format_docs(docs):
    return "\n".join(f"<doc{i+1}>:\nTitle:{doc.metadata['title']}\nSource:{doc.metadata['source']}\nContent:{doc.page_content}\n</doc{i+1}>\n" for i, doc in enumerate(docs))

# Chain
rag_chain = prompt | llm | StrOutputParser()

# Run
generation = rag_chain.invoke({"documents":format_docs(docs_to_use), "question": question})
print(generation)

The four core agentic design patterns are:

1. **Reflection** – the LLM critiques and improves its own output in an iterative loop.  
2. **Tool Use** – the LLM calls external functions (web search, code execution, APIs, etc.) to gather information or perform actions.  
3. **Planning** – the LLM autonomously decomposes a complex goal into a sequence of sub‑steps or tool calls before acting.  
4. **Multi‑Agent Collaboration** – multiple LLM‑based agents, each with a distinct role, work together, exchanging messages and tools to solve a larger task.


## Check for Hallucinations

In [32]:
# Data model
class GradeHallucinations(BaseModel):
    """Binary score for hallucination present in 'generation' answer."""

    binary_score: str = Field(
        ...,
        description="Answer is grounded in the facts, 'yes' or 'no'"
    )

# LLM with function call
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
structured_llm_grader = llm.with_structured_output(GradeHallucinations)

# Prompt
system = """You are a grader assessing whether an LLM generation is grounded in / supported by a set of retrieved facts. \n 
    Give a binary score 'yes' or 'no'. 'Yes' means that the answer is grounded in / supported by the set of facts."""
hallucination_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Set of facts: \n\n <facts>{documents}</facts> \n\n LLM generation: <generation>{generation}</generation>"),
    ]
)

hallucination_grader = hallucination_prompt | structured_llm_grader

response = hallucination_grader.invoke({"documents": format_docs(docs_to_use), "generation": generation})
print(response)

binary_score='yes'


## Highlight used docs

In [ ]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq


# Data model

class HighlightDocuments(BaseModel):
    """Return the specific parts of documents used to answer the question."""

    id: List[str] = Field(
        ...,
        description="List of IDs of documents used to answer the question"
    )

    title: List[str] = Field(
        ...,
        description="List of titles of documents used to answer the question"
    )

    source: List[str] = Field(
        ...,
        description="List of sources of documents used to answer the question"
    )

    segment: List[str] = Field(
        ...,
        description="List of exact verbatim segments from documents that support the answer"
    )


# LLM

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
)

structured_llm = llm.with_structured_output(
    HighlightDocuments
)


# Prompt

system = """
You are an advanced assistant for document search and retrieval.

You are given:

1. A user question.
2. A generated answer.
3. A set of documents that were retrieved and potentially used
   to generate the answer.

Your task is to identify which documents actually support
the generated answer.

For every document that supports the answer:

- Return its ID.
- Return its title.
- Return its source.
- Return an exact verbatim segment from that document that
  supports the answer.

Important rules:

- The segment MUST be copied exactly from the document.
- Do NOT rewrite the segment.
- Do NOT summarize the segment.
- Do NOT modify the wording.
- Do NOT include a document if it does not support the answer.
- Only return documents that directly support the generated answer.

Documents:

{documents}

User question:

{question}

Generated answer:

{generation}
"""


prompt = PromptTemplate(
    template=system,
    input_variables=[
        "documents",
        "question",
        "generation",
    ],
)


# Chain


doc_lookup = prompt | structured_llm


# Run

lookup_response = doc_lookup.invoke({
    "documents": format_docs(docs_to_use),
    "question": question,
    "generation": generation,
})


# Display results

for id, title, source, segment in zip(
    lookup_response.id,
    lookup_response.title,
    lookup_response.source,
    lookup_response.segment,
):
    print(f"ID: {id}")
    print(f"Title: {title}")
    print(f"Source: {source}")
    print(f"Text Segment: {segment}")
    print("-" * 80)

ID: doc1
Title: Four AI Agent Strategies That Improve GPT-4 and GPT-3.5 Performance
Source: https://www.deeplearning.ai/the-batch/how-agents-can-improve-llm-performance/?ref=dl-staging-website.ghost.io
Text Segment: Reflection: The LLM examines its own work to come up with ways to improve it. Tool Use: The LLM is given tools such as web search, code execution, or any other function to help it gather information, take action, or process data. Planning: The LLM comes up with, and executes, a multistep plan to achieve a goal (for example, writing an outline for an essay, then doing online research, then writing a draft, and so on). Multi-agent collaboration: More than one AI agent work together, splitting up tasks and discussing and debating ideas, to come up with better solutions than a single agent would.
--------------------------------------------------------------------------------
ID: doc2
Title: Agentic Design Patterns Part 2: Reflection
Source: https://www.deeplearning.ai/the-batc